# Análise Exploratória dos Dados

</br>

<img src="image/python_supermercado_2025.jpg" alt="Descrição da imagem" width="80%">

O objetivo deste notebook é realizar a Análise Exploratória de Dados (EDA - Exploratory Data Analysis) da base de dados.

<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">
<img src="/Workspace/Users/renanscavazzini@gmail.com/github/modelos/model_supermarket/image/hr-python.png" width="6.5%">

## 1. Configura as bibliotecas

In [0]:
from utils.eda import *

# Suprimir avisos do Pandas
warnings.filterwarnings("ignore")

## 2. Definição dos Parâmetros

Abaixo estão os parâmetros que podem ser ajustados conforme necessário.

In [0]:
tic_geral = time.time()
tic = time.time()

# Diretório:
workspace = "workspace.model_supermarket"

# Caminhos para ler:
df_file = f"{workspace}.df"

# Caminhos para salvar:

# Tipos dos dados:
var_tipos = {
    "chave": [
        "CHAVE_ANON",
    ],
    "data": [
        "DATA",
    ],
    "binario": [],
    "categorico": [
        "PERIODO",
        "CAT_PRODUTO",
    ],
    "string": [
        "SUPERMERCADO",
        "PRODUTO",
        "UNIDADE",
    ],
    "inteiro": [
        "COD_PRODUTO",
    ],
    "numerico": [
        "VALOR_UNIDADE",
        "QTDE",
        "VALOR_TOTAL",
    ],
}
var_tipos

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 3. Leitura da base de dados

In [0]:
tic = time.time()

df = spark.read.format("delta").table(df_file).toPandas()
display(df)

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 3 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 4. Categorização

Na construção da base de dados foram realizadas duas categorizações:

1. Período do dia
2. Categoria do produto

Vamos verificar a volumetria de cada categoria dessas variáveis.

In [0]:
tic = time.time()

display(df.groupby("PERIODO")["PRODUTO"].count())

In [0]:
display(df.groupby("CAT_PRODUTO")["PRODUTO"].count())

# df[(df['CAT_PRODUTO'] == 'OUTROS') & (df['VALOR_TOTAL'] > 0)]

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 4 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 5. Tratamento de inconsistências

In [0]:
tic = time.time()

### 5.1 Tipos de dados

Garantindo que a base de dados estão do tipo correto para cada variável.

In [0]:
# Garantindo que os tipos dos dados estão corretos
for tipo, colunas in var_tipos.items():
    for coluna in colunas:
        if tipo == "data":
            df[coluna] = pd.to_datetime(df[coluna], format="%d/%m/%Y", errors="coerce").dt.date
        elif tipo == "binario":
            df[coluna] = df[coluna].astype("bool")
        elif tipo == "categorico":
            df[coluna] = df[coluna].astype("category")
        elif tipo == "string":
            df[coluna] = df[coluna].astype(str)
        elif tipo == "inteiro":
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("Int64")
        elif tipo == "numerico":
            df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

display(df.dtypes)

In [0]:
display(df.head())

### 5.2 Produtos únicos

A base de dados contém uma chave única que representa cada produto do supermercado: o código do produto. No entanto, os nomes associados a esses produtos podem mudar ao longo do tempo. O ideal é que exista apenas um nome para cada código. Para garantir que não haja esse tipo de inconsistência, será utilizado o nome mais recente de cada produto.

Além disso, pode ser que um produto tenha códigos diferentes para o mesmo nome, pode ser que seja por exemplo um produto com sabor diferente, mas o nome não específica isso. Essa inconsistência será corrijida substrituindo o código do produto pelo código maior (mais recente), pois no final das contas o produto acaba sendo o mesmo.

In [0]:
df = produtos_unicos(df)

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 5 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 6. Propriedades

In [0]:
tic = time.time()

In [0]:
resumo_valores_gerais(df)

In [0]:
resumo_valores_lugar_periodo(df)

In [0]:
resumo_valores_ano_mes(df)

In [0]:
def resumo_produto_mais_comprado(df, quantidade=10, data_inicio=None, data_fim=None):
    """
    Descrição:
        Exibe uma tabela com o top N produtos mais comprados (incluindo empates na última posição),
        com as colunas PRODUTO, QUANTIDADE, TOTAL_GASTO (arredondado em 2 casas decimais).
    --------------------
    Argumentos:
        df (DataFrame): DataFrame contendo os dados dos gastos.
        quantidade (int): Quantidade de produtos a exibir (padrão: 10).
        data_inicio (str): Data de início no formato 'YYYY-MM-DD' (padrão: None).
        data_fim (str): Data de fim no formato 'YYYY-MM-DD' (padrão: None).
    --------------------
    Retorno:
        None
    --------------------
    Autor: 
        Renan Douglas Floriano Scavazzini <renanscavazzini@gmail.com>
    --------------------
    Data da última atualização: 
        04/07/2025
    """
    if data_inicio:
        df = df[df["DATA"] >= pd.to_datetime(data_inicio)]
    if data_fim:
        df = df[df["DATA"] <= pd.to_datetime(data_fim)]

    produtos_agg = (
        df.groupby("PRODUTO")
          .agg(
              QUANTIDADE=("QTDE", "sum"),
              TOTAL_GASTO=("VALOR_TOTAL", "sum")
          )
          .reset_index()
    )
    produtos_agg["TOTAL_GASTO"] = produtos_agg["TOTAL_GASTO"].round(2)
    produtos_agg = produtos_agg.sort_values(by=["QUANTIDADE", "TOTAL_GASTO"], ascending=False)

    produtos_top = produtos_agg.head(quantidade)
    min_quantidade = produtos_top["QUANTIDADE"].min()

    resultado = produtos_agg[produtos_agg["QUANTIDADE"] >= min_quantidade]
    display(resultado[["PRODUTO", "QUANTIDADE", "TOTAL_GASTO"]])

In [0]:
resumo_produto_mais_comprado(df, 10, '2025-01-01', None)

In [0]:
resumo_produto_mais_comprado(df, 10, None, '2022-12-31')

In [0]:
exibe_subtabela_num_produto(df, 10)

In [0]:
exibe_subtabela_nome_produto(df, "whisky")

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 6 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 7. Análise gráfica

In [0]:
tic = time.time()

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 7 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 8. Estudo de domínio

In [0]:
tic = time.time()

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 8 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 9. Análise de outliers

In [0]:
tic = time.time()

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 9 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 10. Transformações

In [0]:
tic = time.time()

In [0]:
toc = time.time()
print(f"\n\033[33mEtapa 10 | Tempo:\033[0;0m {tictoc(tic, toc)}")

## 11. Tempo decorrido

In [0]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")

Previsão de Demanda: Use técnicas de séries temporais para prever a demanda futura de cada produto, permitindo ao supermercado ajustar seus estoques com mais eficiência e reduzir desperdícios.

Recomendação de Produtos: Desenvolva um sistema de recomendação personalizado baseado nos hábitos de compra dos clientes, sugerindo produtos que eles possam estar interessados em comprar.

Análise de Cesta de Compras: Utilize algoritmos de associação para identificar padrões de compra, como quais produtos são frequentemente comprados juntos. Isso pode ajudar a criar promoções ou combos de produtos.

Classificação de Produtos: Automatize a classificação de novos produtos com base nas descrições e características, facilitando a organização do inventário.

Análise de Sentimento: Se houver dados de feedback dos clientes, você pode aplicar técnicas de processamento de linguagem natural (NLP) para analisar o sentimento e entender melhor as opiniões dos clientes sobre os produtos.

Detecção de Anomalias: Use técnicas de detecção de anomalias para identificar transações suspeitas ou inconsistências nos dados, ajudando a detectar fraudes ou erros de entrada de dados.

Segmentação de Clientes: Aplique clustering para segmentar clientes com base em seus comportamentos de compra, permitindo campanhas de marketing mais direcionadas e eficazes.

Análise de Preços: Utilize machine learning para analisar a elasticidade de preços e identificar o impacto de mudanças de preços na demanda dos produtos.

